# 02 — Original DifIISR Baseline Evaluation

This notebook establishes a frozen baseline for the **original DifIISR** model on our corrected thermal-drone dataset. No target-aware modification is applied here.

**Primary goal:** produce reproducible baseline outputs and per-image metrics that can later be compared, image-by-image, with our proposed target-aware method.

Official DifIISR evaluation metrics: **CLIP-IQA, MUSIQ, NIQE, PSNR, LPIPS, SSIM**. PSNR and SSIM follow the official implementation (`test_y_channel=True`, `color_space='ycbcr'`).

Our dataset-specific extension stores per-image values and joins corrected target-size/background metadata for stratified analysis. This extension does not change the DifIISR model.

## 1. Configuration

Use a GPU runtime. Keep `EVAL_SPLIT='valid'` while developing the pipeline. Run the closed `test` split only after the evaluation procedure is frozen.

`DATASET_ROOT` must contain the original thermal images in split folders. Corrected YOLO labels are read from this project's GitHub repository.

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil

PROJECT_REPO = 'https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
DIFIISR_REPO = 'https://github.com/zirui0625/DifIISR.git'
DIFIISR_COMMIT = '09ca97ea48d481656dd8090e84099c963059ac41'

EVAL_SPLIT = 'valid'
SCALE = 4
SEED = 12345
MAX_IMAGES = None  # full validation set

DATASET_ROOT = Path('/content/drive/MyDrive/thermal-drone-dataset')
WORK = Path('/content/difiisr_baseline')
HR_DIR = WORK / EVAL_SPLIT / 'HR'
LR_DIR = WORK / EVAL_SPLIT / 'LR'
SR_DIR = WORK / EVAL_SPLIT / 'SR'
RESULTS_DIR = WORK / 'results'
for p in [HR_DIR, LR_DIR, SR_DIR, RESULTS_DIR]: p.mkdir(parents=True, exist_ok=True)
print('Split:', EVAL_SPLIT, '| Scale:', SCALE, '| Seed:', SEED, '| Full split:', MAX_IMAGES is None)


## 2. Mount Drive and fetch frozen code

DifIISR is pinned to a specific commit so future repository changes cannot silently alter the baseline. The project repository supplies the corrected YOLO annotations and background metadata.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

def run(cmd, cwd=None):
    print('$', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

project_dir = Path('/content/target-aware-infrared-sr')
difiisr_dir = Path('/content/DifIISR')
if not project_dir.exists(): run(['git','clone',PROJECT_REPO,project_dir])
if not difiisr_dir.exists(): run(['git','clone',DIFIISR_REPO,difiisr_dir])
run(['git','fetch','--all'], cwd=difiisr_dir)
run(['git','checkout','--detach',DIFIISR_COMMIT], cwd=difiisr_dir)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=difiisr_dir,text=True).strip() == DIFIISR_COMMIT
print('Pinned DifIISR commit:', DIFIISR_COMMIT)


## 3. Isolated DifIISR Python 3.10 environment

The official repository pins PyTorch 2.1.1 and xformers 0.0.23. We keep these dependencies outside the Colab notebook kernel and execute DifIISR through the isolated interpreter.

In [ ]:
# Create the isolated environment used by the pinned DifIISR repository.
run([sys.executable,'-m','pip','install','-q','uv'])
ENV = Path('/content/difiisr-env')
PY = str(ENV / 'bin' / 'python')
if not Path(PY).exists(): run([sys.executable,'-m','uv','venv','--python','3.10',ENV])
core = ['torch==2.1.1','torchvision==0.16.1','xformers==0.0.23','numpy==1.25.2','scipy==1.9.3','packaging==24.2','setuptools==69.5.1']
extra = ['pyiqa==0.1.12','basicsr==1.4.2','opencv-python','matplotlib','timm','pandas','pillow','scikit-learn','scikit-image','lpips','loguru','omegaconf','six','tqdm','albumentations','einops','imageio','gdown']
run([sys.executable,'-m','uv','pip','install','--python',PY,*core])
run([sys.executable,'-m','uv','pip','install','--python',PY,*extra])
env = os.environ.copy(); env['MPLBACKEND'] = 'Agg'
check = subprocess.run([PY,'-c','import torch, pyiqa, basicsr; print(torch.__version__, torch.cuda.is_available()); print("ENV_OK")'],capture_output=True,text=True,env=env)
print(check.stdout); print(check.stderr)
assert check.returncode == 0, 'DifIISR environment check failed'


## 4. Checkpoint

The official inference code expects `DifIISR/weights/DifIISR.pth`. The autoencoder weight is downloaded automatically by the official code. The cell below uses the checkpoint ID already used in this project; if the file is already present it is not downloaded again.

In [ ]:
weights = difiisr_dir / 'weights'
weights.mkdir(exist_ok=True)
ckpt = weights / 'DifIISR.pth'
if not ckpt.exists():
    run([PY,'-m','gdown','1PhRvk1Dlp3CCrPkrxRfNbNZ3fNgviDVZ','-O',ckpt])
assert ckpt.exists() and ckpt.stat().st_size > 1_000_000, 'DifIISR checkpoint missing or incomplete'
print('Checkpoint:', ckpt, f'({ckpt.stat().st_size/1e6:.1f} MB)')


## 5. Build the full frozen HR validation set

The original thermal validation images are used as HR references. Images are converted to grayscale and cropped only when needed so both dimensions are divisible by ×4. The full validation split is used; no smoke-test limit is applied.


In [ ]:
from PIL import Image
import pandas as pd

label_dir = project_dir / 'annotations' / 'corrected_yolo' / EVAL_SPLIT / 'labels'
assert label_dir.exists(), label_dir
candidate_dirs = [DATASET_ROOT/EVAL_SPLIT/'images', DATASET_ROOT/EVAL_SPLIT]
image_dir = next((p for p in candidate_dirs if p.exists()), None)
assert image_dir is not None, f'No image directory found under {DATASET_ROOT/EVAL_SPLIT}'
exts = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
images = sorted([p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in exts])
if MAX_IMAGES is not None: images = images[:MAX_IMAGES]
assert images, 'No images found'

# Remove stale HR outputs so the directory exactly matches this run.
for p in HR_DIR.glob('*'):
    if p.is_file(): p.unlink()

manifest=[]
for i,src in enumerate(images,1):
    im = Image.open(src).convert('L')
    w,h = im.size
    w4,h4 = (w//SCALE)*SCALE, (h//SCALE)*SCALE
    assert w4 > 0 and h4 > 0
    hr = im.crop((0,0,w4,h4))
    out_name = src.stem + '.png'
    hr.save(HR_DIR/out_name)
    manifest.append({'filename':src.name,'eval_name':out_name,'split':EVAL_SPLIT,'hr_width':w4,'hr_height':h4})
    if i % 250 == 0 or i == len(images): print(f'Prepared {i}/{len(images)}')
manifest = pd.DataFrame(manifest)
manifest.to_csv(RESULTS_DIR/f'{EVAL_SPLIT}_manifest.csv',index=False)
assert len(list(HR_DIR.glob('*.png'))) == len(manifest)
print('Full HR validation set ready:',len(manifest))
display(manifest.head())


## 6. Create LR with the official DifIISR degradation pipeline

DifIISR training does not use simple bicubic downsampling. Its official `trainer.py` creates LQ/LR images with a two-stage Real-ESRGAN-style synthetic degradation: blur kernels, random resizing, Gaussian/Poisson noise, JPEG compression, optional second degradation, final sinc filtering, and a final ×4 size constraint.

This cell reuses the **official repository implementation and `configs/DifIISR_train.yaml` parameters** rather than approximating the pipeline. A fixed seed makes this evaluation set reproducible. The resulting LR files are saved and must be reused unchanged for all future baseline and target-aware comparisons.

> Note: the authors' public test command consumes already-prepared LR/HR pairs; it does not publish a test-time LR-generation command. Therefore this is a controlled adaptation for our HR-only dataset, based directly on their official training degradation.

In [ ]:
# Create a deterministic LR evaluation set using DifIISR's
# Real-ESRGAN-style training degradation components.

from pathlib import Path
import subprocess
import os
import cv2

degrade_script = RESULTS_DIR / "make_official_degradation.py"

degrade_script.write_text(r'''
import argparse
import random
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf

from basicsr.data.degradations import (
    circular_lowpass_kernel,
    random_mixed_kernels,
    random_add_gaussian_noise_pt,
    random_add_poisson_noise_pt,
)
from basicsr.utils import DiffJPEG
from basicsr.utils.img_process_util import filter2D


# ---------------------------------------------------------
# Arguments and configuration
# ---------------------------------------------------------

ap = argparse.ArgumentParser()
ap.add_argument("--hr")
ap.add_argument("--lr")
ap.add_argument("--config")
ap.add_argument("--seed", type=int)
args = ap.parse_args()

cfg = OmegaConf.load(args.config)
d = cfg.degradation
ds = cfg.data.train.params

random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)

device = "cuda"

jpeger = DiffJPEG(differentiable=False).to(device)

pulse = torch.zeros(21, 21, dtype=torch.float32)
pulse[10, 10] = 1


# ---------------------------------------------------------
# Kernel generation
# ---------------------------------------------------------

def kernel_pair():

    k1 = random_mixed_kernels(
        ds.kernel_list,
        ds.kernel_prob,
        ds.blur_kernel_size,
        ds.blur_sigma,
        ds.blur_sigma,
        [-np.pi, np.pi],
        ds.betag_range,
        ds.betap_range,
        noise_range=None,
    )

    pad = (21 - ds.blur_kernel_size) // 2
    k1 = np.pad(k1, ((pad, pad), (pad, pad)))

    if random.random() < ds.sinc_prob:
        omega = np.random.uniform(np.pi / 3, np.pi)
        k1 = circular_lowpass_kernel(
            omega,
            ds.blur_kernel_size,
            pad_to=21,
        )

    k2 = random_mixed_kernels(
        ds.kernel_list2,
        ds.kernel_prob2,
        ds.blur_kernel_size2,
        ds.blur_sigma2,
        ds.blur_sigma2,
        [-np.pi, np.pi],
        ds.betag_range2,
        ds.betap_range2,
        noise_range=None,
    )

    pad = (21 - ds.blur_kernel_size2) // 2
    k2 = np.pad(k2, ((pad, pad), (pad, pad)))

    if random.random() < ds.sinc_prob2:
        omega = np.random.uniform(np.pi / 3, np.pi)
        k2 = circular_lowpass_kernel(
            omega,
            ds.blur_kernel_size2,
            pad_to=21,
        )

    if random.random() < ds.final_sinc_prob:
        ksize = random.choice(list(range(7, 22, 2)))
        omega = np.random.uniform(np.pi / 3, np.pi)

        ks = circular_lowpass_kernel(
            omega,
            ksize,
            pad_to=21,
        )
    else:
        ks = pulse.numpy()

    return [
        torch.tensor(
            x,
            dtype=torch.float32,
            device=device,
        ).unsqueeze(0)
        for x in (k1, k2, ks)
    ]


# ---------------------------------------------------------
# Degradation
# ---------------------------------------------------------

def degrade(im):

    # Thermal images are grayscale.
    # DiffJPEG requires three channels, so replicate the
    # grayscale channel to RGB without adding color information.
    x = torch.from_numpy(im).float().to(device) / 255.0

    x = x.unsqueeze(0).unsqueeze(0)   # [1, 1, H, W]
    x = x.repeat(1, 3, 1, 1)         # [1, 3, H, W]

    oh, ow = x.shape[-2:]
    sf = int(d.sf)

    k1, k2, ks = kernel_pair()

    # First degradation
    out = filter2D(x, k1)

    typ = random.choices(
        ["up", "down", "keep"],
        d.resize_prob,
    )[0]

    if typ == "up":
        sc = random.uniform(1, d.resize_range[1])
    elif typ == "down":
        sc = random.uniform(d.resize_range[0], 1)
    else:
        sc = 1

    out = F.interpolate(
        out,
        scale_factor=sc,
        mode=random.choice(
            ["area", "bilinear", "bicubic"]
        ),
    )

    if random.random() < d.gaussian_noise_prob:
        out = random_add_gaussian_noise_pt(
            out,
            sigma_range=d.noise_range,
            clip=True,
            rounds=False,
            gray_prob=d.gray_noise_prob,
        )
    else:
        out = random_add_poisson_noise_pt(
            out,
            scale_range=d.poisson_scale_range,
            gray_prob=d.gray_noise_prob,
            clip=True,
            rounds=False,
        )

    q = out.new_zeros(out.size(0)).uniform_(
        *d.jpeg_range
    )

    out = jpeger(
        torch.clamp(out, 0, 1),
        quality=q,
    )

    # Second degradation
    if random.random() < d.second_order_prob:

        if random.random() < d.second_blur_prob:
            out = filter2D(out, k2)

        typ = random.choices(
            ["up", "down", "keep"],
            d.resize_prob2,
        )[0]

        if typ == "up":
            sc = random.uniform(
                1,
                d.resize_range2[1],
            )
        elif typ == "down":
            sc = random.uniform(
                d.resize_range2[0],
                1,
            )
        else:
            sc = 1

        out = F.interpolate(
            out,
            size=(
                int(oh / sf * sc),
                int(ow / sf * sc),
            ),
            mode=random.choice(
                ["area", "bilinear", "bicubic"]
            ),
        )

        if random.random() < d.gaussian_noise_prob2:
            out = random_add_gaussian_noise_pt(
                out,
                sigma_range=d.noise_range2,
                clip=True,
                rounds=False,
                gray_prob=d.gray_noise_prob2,
            )
        else:
            out = random_add_poisson_noise_pt(
                out,
                scale_range=d.poisson_scale_range2,
                gray_prob=d.gray_noise_prob2,
                clip=True,
                rounds=False,
            )

    # Final resize + sinc + JPEG
    if random.random() < 0.5:

        out = F.interpolate(
            out,
            size=(oh // sf, ow // sf),
            mode=random.choice(
                ["area", "bilinear", "bicubic"]
            ),
        )

        out = filter2D(out, ks)

        q = out.new_zeros(out.size(0)).uniform_(
            *d.jpeg_range2
        )

        out = jpeger(
            torch.clamp(out, 0, 1),
            quality=q,
        )

    else:

        q = out.new_zeros(out.size(0)).uniform_(
            *d.jpeg_range2
        )

        out = jpeger(
            torch.clamp(out, 0, 1),
            quality=q,
        )

        out = F.interpolate(
            out,
            size=(oh // sf, ow // sf),
            mode=random.choice(
                ["area", "bilinear", "bicubic"]
            ),
        )

        out = filter2D(out, ks)

    out = torch.clamp(
        (out * 255.0).round(),
        0,
        255,
    )

    # All three channels started identical. Convert the
    # degraded result back to one grayscale channel.
    out = out[0].mean(dim=0)

    return out.byte().cpu().numpy()


# ---------------------------------------------------------
# Process HR images
# ---------------------------------------------------------

hr = Path(args.hr)
lr = Path(args.lr)

lr.mkdir(parents=True, exist_ok=True)

# Remove previous partial outputs.
for old_file in lr.glob("*.png"):
    old_file.unlink()

files = sorted(hr.glob("*.png"))

if not files:
    raise RuntimeError(
        f"No HR PNG images found in {hr}"
    )

for i, p in enumerate(files):

    im = cv2.imread(
        str(p),
        cv2.IMREAD_GRAYSCALE,
    )

    if im is None:
        raise RuntimeError(
            f"Could not read image: {p}"
        )

    out = degrade(im)

    ok = cv2.imwrite(
        str(lr / p.name),
        out,
    )

    if not ok:
        raise RuntimeError(
            f"Could not save LR image: {p.name}"
        )

    print(
        f"{i + 1}/{len(files)} "
        f"{p.name}: "
        f"{im.shape[1]}x{im.shape[0]} -> "
        f"{out.shape[1]}x{out.shape[0]}"
    )

print(
    "OFFICIAL_DEGRADATION_DONE",
    len(files),
)
''')


# ---------------------------------------------------------
# Run the degradation script
# ---------------------------------------------------------

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

cmd = [
    PY,
    str(degrade_script),
    "--hr", str(HR_DIR),
    "--lr", str(LR_DIR),
    "--config",
    str(difiisr_dir / "configs" / "DifIISR_train.yaml"),
    "--seed", str(SEED),
]

print("$", " ".join(map(str, cmd)))

result = subprocess.run(
    list(map(str, cmd)),
    cwd=str(difiisr_dir),
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("LR degradation failed.")


# ---------------------------------------------------------
# Validate the frozen LR set
# ---------------------------------------------------------

lr_files = sorted(LR_DIR.glob("*.png"))
hr_files = sorted(HR_DIR.glob("*.png"))

assert len(lr_files) == len(hr_files), (
    f"HR/LR mismatch: {len(hr_files)} HR vs "
    f"{len(lr_files)} LR"
)

for hr_path, lr_path in zip(hr_files, lr_files):

    hr_im = cv2.imread(
        str(hr_path),
        cv2.IMREAD_GRAYSCALE,
    )

    lr_im = cv2.imread(
        str(lr_path),
        cv2.IMREAD_GRAYSCALE,
    )

    expected_h = hr_im.shape[0] // int(SCALE)
    expected_w = hr_im.shape[1] // int(SCALE)

    assert lr_im.shape == (
        expected_h,
        expected_w,
    ), (
        f"Wrong LR size for {lr_path.name}: "
        f"{lr_im.shape}, expected "
        f"{(expected_h, expected_w)}"
    )

print(
    f"✓ Frozen LR set ready: {LR_DIR}"
)
print(
    f"✓ Images: {len(lr_files)}"
)
print(
    f"✓ Scale: x{SCALE}"
)

## 7. Run the original DifIISR baseline

Run the pinned pretrained DifIISR model on the frozen LR validation set. Inference is intentionally run **without `-reference`** because the repository's automatic paired evaluation mixes RGB SR output with grayscale HR references. Paired paper metrics are computed cleanly in the next section with matched RGB tensors.


In [ ]:
# Full validation inference: frozen LR -> original DifIISR -> SR
env = os.environ.copy(); env['MPLBACKEND'] = 'Agg'
for p in SR_DIR.glob('*'):
    if p.is_file(): p.unlink()
cmd=[PY,'inference.py','-input',str(LR_DIR),'-output',str(SR_DIR),'--config','configs/DifIISR_test.yaml','--seed',str(SEED)]
print('$',' '.join(map(str,cmd)))
result=subprocess.run(list(map(str,cmd)),cwd=str(difiisr_dir),env=env)
if result.returncode != 0: raise RuntimeError('DifIISR inference failed; inspect the log above.')
lr_files=sorted(LR_DIR.glob('*.png')); sr_files=sorted(SR_DIR.glob('*.png'))
assert len(sr_files)==len(lr_files), f'LR/SR mismatch: {len(lr_files)} vs {len(sr_files)}'
print(f'✓ DifIISR inference completed: {len(sr_files)} images')


## 8. Paper-style evaluation on our validation set

The paper reports reference-based **PSNR, SSIM and LPIPS**, and no-reference image-quality metrics including **CLIP-IQA, MUSIQ and NIQE**. We compute the same metric family on every SR image and then report dataset averages. Both SR and HR are loaded as three-channel RGB tensors so paired metrics receive matching shapes.


In [ ]:
# Section 9 below computes the six metrics per image and their aggregate mean.
# Keeping evaluation in one implementation avoids the grayscale/RGB mismatch in the repository's evaluate.py.
print('Evaluation will be computed in the next cell for', len(list(SR_DIR.glob('*.png'))), 'SR images.')


## 9. Per-image metrics and aggregate paper-style results

Compute all six metrics per image, save the table, and print the validation-set mean used as the baseline result.


In [ ]:
metric_script = RESULTS_DIR / 'per_image_metrics.py'
metric_script.write_text(r'''
from pathlib import Path
import argparse, pandas as pd, torch, pyiqa
from PIL import Image
import numpy as np
ap=argparse.ArgumentParser(); ap.add_argument('--sr'); ap.add_argument('--hr'); ap.add_argument('--out'); args=ap.parse_args()
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metrics={
 'clipiqa':pyiqa.create_metric('clipiqa').to(device),
 'musiq':pyiqa.create_metric('musiq').to(device),
 'niqe':pyiqa.create_metric('niqe').to(device),
 'psnr':pyiqa.create_metric('psnr',test_y_channel=True,color_space='ycbcr').to(device),
 'lpips':pyiqa.create_metric('lpips').to(device),
 'ssim':pyiqa.create_metric('ssim',test_y_channel=True,color_space='ycbcr').to(device),
}
def tensor(path):
 x=np.asarray(Image.open(path).convert('RGB'),dtype=np.float32)/255.0
 return torch.from_numpy(x).permute(2,0,1).unsqueeze(0).to(device)
sr=Path(args.sr); hr=Path(args.hr); rows=[]
files=sorted(sr.glob('*.png'))
for i,sp in enumerate(files,1):
 hp=hr/sp.name
 if not hp.exists(): raise FileNotFoundError(hp)
 a,b=tensor(sp),tensor(hp); row={'eval_name':sp.name}
 if a.shape != b.shape: raise RuntimeError(f'Shape mismatch for {sp.name}: {a.shape} vs {b.shape}')
 with torch.no_grad():
  for k in ['clipiqa','musiq','niqe']: row[k]=float(metrics[k](a).item())
  for k in ['psnr','lpips','ssim']: row[k]=float(metrics[k](a,b).item())
 rows.append(row)
 if i % 100 == 0 or i == len(files): print(f'Evaluated {i}/{len(files)}')
df=pd.DataFrame(rows); df.to_csv(args.out,index=False)
print('\n=== VALIDATION MEAN ===')
print(df[['clipiqa','musiq','niqe','psnr','lpips','ssim']].mean().to_string())
print('\nSaved',args.out)
''')
per_image_csv = RESULTS_DIR/f'{EVAL_SPLIT}_difiisr_per_image_metrics.csv'
env = os.environ.copy(); env['MPLBACKEND']='Agg'
subprocess.run([PY,str(metric_script),'--sr',str(SR_DIR),'--hr',str(HR_DIR),'--out',str(per_image_csv)],cwd=str(difiisr_dir),env=env,check=True)
metrics_df=pd.read_csv(per_image_csv)
display(metrics_df.head())


## 10. Join corrected target metadata

This section derives target-size strata from the frozen corrected YOLO labels. Background-only images are identified by empty label files. This does not affect the official metrics; it only lets us ask where the baseline succeeds or fails.

In [ ]:
import numpy as np
metrics_df = pd.read_csv(per_image_csv)
df = manifest.merge(metrics_df,on='eval_name',how='inner')

def label_stats(row):
    lp = label_dir / (Path(row.filename).stem + '.txt')
    if not lp.exists() or not lp.read_text().strip(): return pd.Series({'n_targets':0,'max_target_dim_lr':np.nan,'target_group':'background'})
    vals=[]
    for line in lp.read_text().splitlines():
        p=line.split(); bw=float(p[3])*row.hr_width; bh=float(p[4])*row.hr_height
        vals.append(max(bw,bh)/SCALE)
    m=max(vals)
    group = '<=2' if m<=2 else '2-4' if m<=4 else '4-8' if m<=8 else '8-16' if m<=16 else '>16'
    return pd.Series({'n_targets':len(vals),'max_target_dim_lr':m,'target_group':group})
df = pd.concat([df,df.apply(label_stats,axis=1)],axis=1)

bg_path = project_dir/'metadata'/'background_metadata.csv'
if bg_path.exists():
    bg=pd.read_csv(bg_path)[['filename','background_cluster','background_type']]
    df=df.merge(bg,on='filename',how='left')

out = RESULTS_DIR/f'{EVAL_SPLIT}_difiisr_baseline.csv'
df.to_csv(out,index=False)
print('Saved:',out)
display(df.head())


## 11. Baseline performance by target size and background type

These tables are the main diagnostic extension for our research question. They do **not** replace the official aggregate DifIISR metrics.

In [ ]:
metric_cols=['clipiqa','musiq','niqe','psnr','lpips','ssim']
size_summary=df.groupby('target_group')[metric_cols].agg(['mean','std','count'])
display(size_summary)
size_summary.to_csv(RESULTS_DIR/f'{EVAL_SPLIT}_metrics_by_target_size.csv')

if 'background_type' in df.columns:
    bg_summary=df[df.target_group=='background'].groupby('background_type')[metric_cols].agg(['mean','std','count'])
    display(bg_summary)
    bg_summary.to_csv(RESULTS_DIR/f'{EVAL_SPLIT}_metrics_by_background_type.csv')


## 12. Freeze the baseline record

Before final test evaluation, save the exact commit, seed, scale factor, image manifest and result tables. Future target-aware experiments must reuse the same HR/LR pairs and evaluation definitions.

**Interpretation rule:** the current notebook establishes how original DifIISR performs on our data. Only after this baseline is frozen should the proposed target-aware guidance be evaluated against it.

In [ ]:
record={'method':'Original DifIISR','difiisr_commit':DIFIISR_COMMIT,'split':EVAL_SPLIT,'scale':SCALE,'seed':SEED,'n_images':len(df),'metrics':['CLIP-IQA','MUSIQ','NIQE','PSNR(Y)','LPIPS','SSIM(Y)']}
(RESULTS_DIR/f'{EVAL_SPLIT}_baseline_record.json').write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))
print('\nBaseline artifacts:', RESULTS_DIR)
